# Experimento 3: Diferencias temáticas por tipo de sesión

**Pregunta:** ¿Existen diferencias temáticas entre sesiones ordinarias, especiales y extraordinarias?

**Input:** Modelo BERTopic (`data/bertopic_diputados_final`) + tópicos asignados (`data/intervenciones_con_topico.parquet`) + metadata de sesiones (`data/parquets/`)  
**Output:** Heatmap de prevalencia por tipo de sesión, tests estadísticos (Kruskal-Wallis + Dunn), violin plots de tópicos discriminantes

## 1. Imports y carga del modelo

In [24]:
import pandas as pd
import numpy as np
import glob
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from scipy import stats
from scikit_posthocs import posthoc_dunn
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Cargando modelo BERTopic...")
topic_model = BERTopic.load("../data/bertopic_diputados_final", embedding_model=embedding_model)
print(f"Modelo cargado: {len(topic_model.get_topic_info()) - 1} tópicos")

Cargando modelo BERTopic...
Modelo cargado: 190 tópicos


## 2. Cargar datos e incorporar tipo de sesión

In [25]:
# ── Metadata de sesiones (tipo) desde parquets por período ────────────────────
dfs_meta = []
for f in glob.glob("../data/parquets/*.parquet"):
    df_tmp = pd.read_parquet(f, columns=["id_periodo", "id_reunion", "descripcion"])
    dfs_meta.append(df_tmp)

meta = pd.concat(dfs_meta, ignore_index=True).drop_duplicates(
    subset=["id_periodo", "id_reunion"]
)

def extraer_tipo(desc):
    if not isinstance(desc, str):
        return "Otra"
    d = desc.lower()
    if "asamblea legislativa" in d:        return "Asamblea Legislativa"
    if "expresion" in d or "minor" in d:   return "Expresión en Minoría"
    if "extraordinaria especial" in d:     return "Extraordinaria Especial"
    if "extraordinaria" in d:              return "Extraordinaria"
    if "especial" in d:                    return "Especial"
    if "ordinaria" in d:                   return "Ordinaria"
    if "preparatoria" in d:                return "Preparatoria"
    if "informativa" in d:                 return "Informativa"
    return "Otra"

meta["tipo_sesion"] = meta["descripcion"].apply(extraer_tipo)

# ── Intervenciones limpias ────────────────────────────────────────────────────
print("Cargando intervenciones...")
df = pd.read_parquet("../data/intervenciones_limpias.parquet")
df["fecha"] = pd.to_datetime(df["fecha"])
df["anio"] = df["fecha"].dt.year
df = df.dropna(subset=["texto_limpio"])
df = df[df["texto_limpio"].str.strip() != ""]

# Unir tipo de sesión
df = df.merge(
    meta[["id_periodo", "id_reunion", "tipo_sesion"]],
    on=["id_periodo", "id_reunion"],
    how="left"
)

print(f"Intervenciones cargadas: {len(df)}")
print(f"Cobertura tipo_sesion: {df['tipo_sesion'].notna().mean():.1%}")

Cargando intervenciones...
Intervenciones cargadas: 172417
Cobertura tipo_sesion: 100.0%


## 3. EDA: distribución por tipo de sesión

Antes de analizar tópicos, revisamos cuántas sesiones e intervenciones hay por tipo para decidir con cuáles trabajar.

In [26]:
# ── Tabla resumen ─────────────────────────────────────────────────────────────
sesiones_por_tipo = meta["tipo_sesion"].value_counts().rename("sesiones")
interv_por_tipo   = df["tipo_sesion"].value_counts().rename("intervenciones")

resumen = pd.concat([sesiones_por_tipo, interv_por_tipo], axis=1).fillna(0).astype(int)
resumen["interv_por_sesion"] = (resumen["intervenciones"] / resumen["sesiones"]).round(1)
resumen = resumen.sort_values("intervenciones", ascending=False)
print(resumen.to_string())

                         sesiones  intervenciones  interv_por_sesion
tipo_sesion                                                         
Especial                      219           67388              307.7
Ordinaria                     368           61180              166.2
Extraordinaria Especial        86           16064              186.8
Extraordinaria                 78           12736              163.3
Asamblea Legislativa           52            6056              116.5
Otra                           28            4400              157.1
Expresión en Minoría          197            2710               13.8
Preparatoria                   13            1879              144.5
Informativa                     2               4                2.0


In [ ]:
fig_eda = px.bar(
    resumen.reset_index().rename(columns={"index": "tipo_sesion"}),
    x="tipo_sesion",
    y="intervenciones",
    color="tipo_sesion",
    text="intervenciones",
    title="Intervenciones por tipo de sesión",
    labels={"tipo_sesion": "Tipo de sesión", "intervenciones": "Cantidad de intervenciones"},
    height=500,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_eda.update_traces(textposition="outside")
fig_eda.update_layout(
    plot_bgcolor="white",
    showlegend=False,
    xaxis=dict(categoryorder="total descending"),
)
fig_eda.show()
fig_eda.write_html("figuras/exp3/eda_tipos_sesion.html")

### Decisión: tipos de sesión seleccionados

| Tipo | Sesiones | Intervenciones | Decisión |
|---|---|---|---|
| Ordinaria | 368 | ~89.000 | ✅ Incluir |
| Especial | 219 | ~93.000 | ✅ Incluir |
| Extraordinaria Especial | 86 | ~22.000 | ✅ Incluir |
| Extraordinaria | 78 | ~18.000 | ✅ Incluir |
| Expresión en Minoría | 197 | ~3.600 | ❌ Pocas intervenciones por sesión (~18) |
| Asamblea Legislativa | 52 | ~8.300 | ❌ Actos formales, poco contenido temático |
| Preparatoria | 13 | ~2.300 | ❌ Muy pocas sesiones |
| Informativa | 2 | 4 | ❌ Descartable |
| Otra | 28 | ~6.100 | ❌ Sin sentido semántico claro |

**Nos quedamos con las 4 primeras**, que son las que tienen suficiente masa de datos y representan los tipos de sesión con claro sentido institucional.

## 4. Filtrar tipos de sesión y asignar tópicos

In [28]:
TIPOS_SELECCIONADOS = ["Ordinaria", "Especial", "Extraordinaria Especial", "Extraordinaria"]
df = df[df["tipo_sesion"].isin(TIPOS_SELECCIONADOS)].copy()
print(f"Intervenciones con tipo de sesión válido: {len(df)}")

# ── Asignar tópicos desde caché ───────────────────────────────────────────────
TOPICOS_PATH = Path("../data/intervenciones_con_topico.parquet")
df_topicos = pd.read_parquet(
    TOPICOS_PATH,
    columns=["id_periodo", "id_reunion", "n_intervencion", "topic"]
).drop_duplicates(subset=["id_periodo", "id_reunion", "n_intervencion"])

topico_map = df_topicos.set_index(["id_periodo", "id_reunion", "n_intervencion"])["topic"]
df["topic"] = (
    df.set_index(["id_periodo", "id_reunion", "n_intervencion"])
    .index.map(topico_map)
    .values
)
del df_topicos, topico_map
print(f"Tópico asignado en {df['topic'].notna().mean():.1%} de las intervenciones")

Intervenciones con tipo de sesión válido: 157368
Tópico asignado en 100.0% de las intervenciones


## 5. Selección de tópicos (mismos 10 que Experimento 2)

In [29]:
TOPICOS_SELECCIONADOS = {
    1:   "Presupuesto / Finanzas públicas",
    6:   "Derecho penal",
    8:   "Impuestos / Fiscal",
    11:  "Trabajo / Laboral",
    12:  "Salud / Discapacidad",
    18:  "Jubilaciones / Previsional",
    5:   "Energía / Gas / Combustibles",
    129: "Derechos humanos / Terrorismo",
    30:  "Agropecuario / Ganadería",
    22:  "Defensa / Fuerzas militares",
}
topicos_ids = list(TOPICOS_SELECCIONADOS.keys())
topic_model.set_topic_labels(TOPICOS_SELECCIONADOS)

df_validos = df[df["topic"].isin(topicos_ids)].copy()
df_validos["topic_label"] = df_validos["topic"].map(TOPICOS_SELECCIONADOS)

print(f"Intervenciones con tópico seleccionado: {len(df_validos)}")
print()
print(df_validos.groupby("tipo_sesion")["topic_label"].count().rename("intervenciones"))

Intervenciones con tópico seleccionado: 17478

tipo_sesion
Especial                   6757
Extraordinaria             1497
Extraordinaria Especial    1972
Ordinaria                  7252
Name: intervenciones, dtype: int64


## 6. Heatmap: prevalencia de tópicos por tipo de sesión

In [ ]:
pivot = (
    df_validos
    .groupby(["tipo_sesion", "topic_label"])
    .size()
    .reset_index(name="count")
)
pivot["prop"] = pivot.groupby("tipo_sesion")["count"].transform(lambda x: x / x.sum())

heatmap_df = pivot.pivot(index="topic_label", columns="tipo_sesion", values="prop").fillna(0)

orden_tipos = [t for t in TIPOS_SELECCIONADOS if t in heatmap_df.columns]
heatmap_df = heatmap_df[orden_tipos]

fig_heat = px.imshow(
    heatmap_df,
    labels=dict(x="Tipo de sesión", y="Tópico", color="Proporción"),
    title="Prevalencia de tópicos por tipo de sesión",
    color_continuous_scale="Blues",
    aspect="auto",
    height=600,
)
fig_heat.update_xaxes(tickangle=15)
fig_heat.show()
fig_heat.write_html("figuras/exp3/heatmap_topicos_por_tipo_sesion.html")

## 8. Tests estadísticos: Kruskal-Wallis + Dunn

Para cada tópico, el test de Kruskal-Wallis evalúa si la prevalencia difiere significativamente entre los 4 tipos de sesión (α = 0.05).  
Donde el test resulte significativo, se aplica el test de Dunn como post-hoc para identificar qué pares de tipos difieren.

In [32]:
# Calcular proporción de cada tópico por sesión (unidad de análisis = sesión)
prop_sesion = (
    df_validos
    .groupby(["id_periodo", "id_reunion", "tipo_sesion", "topic_label"])
    .size()
    .reset_index(name="count")
)
total_sesion = (
    df_validos
    .groupby(["id_periodo", "id_reunion"])
    .size()
    .reset_index(name="total")
)
prop_sesion = prop_sesion.merge(total_sesion, on=["id_periodo", "id_reunion"])
prop_sesion["prop"] = prop_sesion["count"] / prop_sesion["total"]

# Kruskal-Wallis por tópico
resultados_kw = []
for topico in TOPICOS_SELECCIONADOS.values():
    grupos = [
        prop_sesion[prop_sesion["tipo_sesion"] == t]
                   [prop_sesion["topic_label"] == topico]["prop"].values
        for t in TIPOS_SELECCIONADOS
    ]
    # Excluir grupos vacíos
    grupos = [g for g in grupos if len(g) > 0]
    if len(grupos) < 2:
        continue
    stat, p = stats.kruskal(*grupos)
    resultados_kw.append({"topico": topico, "H": round(stat, 3), "p_valor": round(p, 5)})

df_kw = pd.DataFrame(resultados_kw).sort_values("p_valor")
df_kw["significativo"] = df_kw["p_valor"] < 0.05
print("Resultados Kruskal-Wallis (α = 0.05):")
print(df_kw.to_string(index=False))

Resultados Kruskal-Wallis (α = 0.05):
                         topico      H  p_valor  significativo
       Agropecuario / Ganadería 22.406  0.00005           True
     Jubilaciones / Previsional 19.281  0.00024           True
           Salud / Discapacidad 10.003  0.01854           True
              Trabajo / Laboral  9.858  0.01982           True
    Defensa / Fuerzas militares  8.236  0.04138           True
Presupuesto / Finanzas públicas  7.390  0.06045          False
                  Derecho penal  7.383  0.06066          False
   Energía / Gas / Combustibles  6.827  0.07763          False
             Impuestos / Fiscal  5.887  0.11722          False
  Derechos humanos / Terrorismo  5.415  0.14381          False


In [33]:
# Dunn's test para los tópicos significativos
topicos_sig = df_kw[df_kw["significativo"]]["topico"].tolist()
print(f"{len(topicos_sig)} tópicos con diferencias significativas entre tipos de sesión:\n")

for topico in topicos_sig:
    datos_topico = prop_sesion[prop_sesion["topic_label"] == topico]
    grupos_dunn = [
        datos_topico[datos_topico["tipo_sesion"] == t]["prop"].values
        for t in TIPOS_SELECCIONADOS
        if len(datos_topico[datos_topico["tipo_sesion"] == t]) > 0
    ]
    labels_dunn = [
        t for t in TIPOS_SELECCIONADOS
        if len(datos_topico[datos_topico["tipo_sesion"] == t]) > 0
    ]
    dunn = posthoc_dunn(grupos_dunn, p_adjust="bonferroni")
    dunn.index   = labels_dunn
    dunn.columns = labels_dunn
    print(f"── {topico} ──")
    print(dunn.round(4).to_string())
    print()

5 tópicos con diferencias significativas entre tipos de sesión:

── Agropecuario / Ganadería ──
                         Ordinaria  Especial  Extraordinaria Especial  Extraordinaria
Ordinaria                   1.0000    0.0005                   0.0044          1.0000
Especial                    0.0005    1.0000                   1.0000          0.5706
Extraordinaria Especial     0.0044    1.0000                   1.0000          0.2217
Extraordinaria              1.0000    0.5706                   0.2217          1.0000

── Jubilaciones / Previsional ──
                         Ordinaria  Especial  Extraordinaria Especial  Extraordinaria
Ordinaria                   1.0000    0.0002                   0.1136          1.0000
Especial                    0.0002    1.0000                   1.0000          0.2655
Extraordinaria Especial     0.1136    1.0000                   1.0000          1.0000
Extraordinaria              1.0000    0.2655                   1.0000          1.0000

── Salud 

## 9. Tópicos discriminantes: proporción por tipo de sesión

Filtramos los tópicos con diferencias significativas (p < 0.05) y repetimos el gráfico de barras solo con ellos para facilitar la lectura.

In [ ]:
topicos_sig = df_kw[df_kw["significativo"]]["topico"].tolist()
print(f"Tópicos con diferencias significativas: {topicos_sig}")

prop_sig = prop_tipo[prop_tipo["topic_label"].isin(topicos_sig)].copy()

fig_sig = px.bar(
    prop_sig,
    x="topic_label",
    y="prop",
    color="tipo_sesion",
    barmode="group",
    title="Proporción por tipo de sesión — tópicos con diferencias significativas",
    labels={
        "topic_label": "Tópico",
        "prop": "Proporción de intervenciones",
        "tipo_sesion": "Tipo de sesión",
    },
    color_discrete_sequence=px.colors.qualitative.Set2,
    height=500,
    width=900,
)
fig_sig.update_xaxes(tickangle=20)
fig_sig.update_layout(
    plot_bgcolor="white",
    legend_title_text="Tipo de sesión",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
)
fig_sig.show()
fig_sig.write_html("figuras/exp3/barras_topicos_significativos_tipo_sesion.html")

## 10. Guardar resultados

In [35]:
df_kw.to_csv("../data/kruskal_wallis_tipo_sesion.csv", index=False)
print("Guardado: data/kruskal_wallis_tipo_sesion.csv")

Guardado: data/kruskal_wallis_tipo_sesion.csv
